### RAG EVALUATION

In [64]:
import os
import csv
from dotenv import load_dotenv 
import pandas as pd
# Allows running asyncio in environments with an existing event loop, like Jupyter notebooks.
import nest_asyncio

#llamaindex
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import Document
from llama_index.core.schema import TextNode
from llama_index.core.node_parser import TokenTextSplitter
from llama_index.core.schema import BaseNode
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core import VectorStoreIndex
from llama_index.llms.google_genai import GoogleGenAI
import google.genai.types as types
from google import genai

#to determine the hash
import hashlib

#Vector Store ChromaDB
import chromadb


#for generating questions - curating golden context dataset
from llama_index.core.llms.utils import LLM
from llama_index.core.schema import MetadataMode, TextNode
from tqdm import tqdm
import json
import re
import uuid
import warnings
import time
from typing import Dict, List, Tuple
from llama_index.core.evaluation import EmbeddingQAFinetuneDataset

#eval retrieval
from llama_index.core.evaluation import RetrieverEvaluator

#eval response 
from llama_index.core.evaluation import RelevancyEvaluator, FaithfulnessEvaluator, BatchEvalRunner
from llama_index.llms.openai import OpenAI

# Create your index
from llama_index.core import VectorStoreIndex
index = VectorStoreIndex.from_vector_store(vector_store)

#correctness metrics 
from llama_index.core.evaluation import CorrectnessEvaluator

In [2]:
# Load variables from .env file
load_dotenv()
nest_asyncio.apply()

In [3]:
# Access the keys
openai_key = os.getenv("OPENAI_API_KEY")
google_key = os.getenv("GOOGLE_API_KEY")

print("Keys loaded successfully")

Keys loaded successfully


In [4]:
#we are setting up the text embedding model 
#so if the user has not explicitly mentioned some other model we will be using the openai text-embedding-3-small embedding model
#this is to make sure that we are using the same embedding model throughout the RAG pipeline
Settings.embed_model = OpenAIEmbedding(
    model = "text-embedding-3-small"
)

In [6]:
#DOWANLOAD THE DATA 14 TOWARDS AI BLOGS
!wget https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles.csv 


--2026-07-22 14:51:13--  https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 173646 (170K) [text/plain]
Saving to: ‘mini-llama-articles.csv’

mini-llama-articles 100%[===================>] 169.58K  --.-KB/s    in 0.08s   

2026-07-22 14:51:14 (1.97 MB/s) - ‘mini-llama-articles.csv’ saved [173646/173646]



### CREATE A VECTOR STORE 

- create a client and a new collection 
- chromadb.EphemeralClient saves data in-memory.

In [7]:
chroma_client = chromadb.PersistentClient(path = "/Users/r31881/Desktop/AI/helloai/data/mini-llama-articles")


In [8]:
chroma_collection = chroma_client.create_collection("mini-llama-articles")

- define a storage context object using the created vector database

In [10]:
vector_store = ChromaVectorStore(chroma_collection = chroma_collection)

- Now lets load the data in rows list

In [9]:
rows = []
#load the file as a json 
with open("/Users/r31881/Desktop/AI/helloai/data/mini-llama-articles.csv",mode='r',encoding='utf-8') as file:
    csv_reader = csv.reader(file)
    for index,row in enumerate(csv_reader):
        if index == 0:
            continue
        rows.append(row)

In [11]:
len(rows)

14

In [12]:
rows[:2]

[["Beyond GPT-4: What's New?",
  'LLM Variants and Meta\'s Open Source Before shedding light on four major trends, I\'d share the latest Meta\'s Llama 2 and Code Llama. Meta\'s Llama 2 represents a sophisticated evolution in LLMs. This suite spans models pretrained and fine-tuned across a parameter spectrum of 7 billion to 70 billion. A specialized derivative, Llama 2-Chat, has been engineered explicitly for dialogue-centric applications. Benchmarking revealed Llama 2\'s superior performance over most extant open-source chat models. Human-centric evaluations, focusing on safety and utility metrics, positioned Llama 2-Chat as a potential contender against proprietary, closed-source counterparts. The development trajectory of Llama 2 emphasized rigorous fine-tuning methodologies. Meta\'s transparent delineation of these processes aims to catalyze community-driven advancements in LLMs, underscoring a commitment to collaborative and responsible AI development. Code Llama is built on top of

- here we will define the text into the documents 
- Convert the chunks to Document objects so the LlamaIndex framework can process them.
- in the document we will store -> article, title, URL, source_name
- document object randomly assign the document id so it is always good to assign the document_id manually to maintain the reproducibility. 
- This is to make sure that doc_1 represents first document only 

In [15]:
documents = [
    Document(
        text=row[1], metadata={"title":row[0],"url":row[2],"source_name":row[3]},
    )
    for row in rows
]
#manaully setting up the document id 
for index, doc in enumerate(documents):
    doc.id_ = f"doc_{index}"

In [16]:
len(documents)

14

In [17]:
documents[:3]

[Document(id_='doc_0', embedding=None, metadata={'title': "Beyond GPT-4: What's New?", 'url': 'https://pub.towardsai.net/beyond-gpt-4-whats-new-cbd61a448eb9#dda8', 'source_name': 'towards_ai'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='LLM Variants and Meta\'s Open Source Before shedding light on four major trends, I\'d share the latest Meta\'s Llama 2 and Code Llama. Meta\'s Llama 2 represents a sophisticated evolution in LLMs. This suite spans models pretrained and fine-tuned across a parameter spectrum of 7 billion to 70 billion. A specialized derivative, Llama 2-Chat, has been engineered explicitly for dialogue-centric applications. Benchmarking revealed Llama 2\'s superior performance over most extant open-source chat models. Human-centric evaluations, focusing on safety and utility metrics, positioned Llama 2-Chat as a 

- which document created which node. we need to create a reproducible/deterministic id for each node. 
- for that we leverage the usage of hashlib and SHA-256 ago
- node + document ==> HEXADECIMAL string 

In [18]:
def deterministic_id_func(i:int,doc:BaseNode)->str:
    """
    Deterministic ID function for the text splitter

    This will be used to generate a unique repeatable identifier for each node.

    """
    unique_identifier = doc.id_ + str(i)
    hasher = hashlib.sha256()
    hasher.update(unique_identifier.encode("utf-8"))
    return hasher.hexdigest()

In [19]:
text_spliter = TokenTextSplitter(
    separator = " ", chunk_size = 512, chunk_overlap = 128, id_func = deterministic_id_func
)

In [20]:
text_spliter

TokenTextSplitter(include_metadata=True, include_prev_next_rel=True, callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x11a28f160>, id_func=<function deterministic_id_func at 0x11a269630>, chunk_size=512, chunk_overlap=128, separator=' ', backup_separators=['\n'], keep_whitespaces=False)

### INGESTION PIPELINE

In [ ]:
"""
pipeline.run(documents=documents)
    │
    ├── Step 1: TokenTextSplitter
    │       → splits Document into chunks
    │       → creates TextNode(id_='4ab5...', text='...', embedding=None)
    │
    ├── Step 2: OpenAIEmbedding
    │       → calls OpenAI API for each node
    │       → fills in embedding=[0.00448, 0.0167, ...]
    │
    └── Step 3: vector_store (Chroma)
            → stores the node WITH its embedding inside Chroma
"""


"\npipeline.run(documents=documents)\n    │\n    ├── Step 1: TokenTextSplitter\n    │       → splits Document into chunks\n    │       → creates TextNode(id_='4ab5...', text='...', embedding=None)\n    │\n    ├── Step 2: OpenAIEmbedding\n    │       → calls OpenAI API for each node\n    │       → fills in embedding=[0.00448, 0.0167, ...]  ✅\n    │\n    └── Step 3: vector_store (Chroma)\n            → stores the node WITH its embedding inside Chroma\n"

In [21]:
pipeline = IngestionPipeline(
    transformations = [
        text_spliter,
        OpenAIEmbedding(model = 'text-embedding-3-small'),
    ],
    vector_store = vector_store,
)

In [22]:
nodes = pipeline.run(documents = documents, show_progress = True)

/Users/r31881/.pyenv/versions/helloai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating embeddings: 100%|██████████| 108/108 [00:04<00:00, 22.01it/s]


In [23]:
len(nodes)

108

In [24]:
nodes[:3]

[TextNode(id_='4ab5bd897f01474fc9b0049f95e31edae3ccd9e74d0f0acd3932b50a74d608b6', embedding=[0.004486083984375, 0.0167083740234375, 0.06158447265625, -0.016204833984375, 0.020477294921875, -0.0224609375, 0.00628662109375, 0.01470947265625, -0.00012314319610595703, 0.005825042724609375, 0.0275115966796875, -0.04571533203125, -0.03533935546875, 0.00426483154296875, -0.035125732421875, -0.0277862548828125, -0.034088134765625, -0.046356201171875, -0.0154266357421875, 0.03765869140625, 0.013092041015625, 0.0072021484375, -0.034515380859375, 0.025909423828125, -0.00504302978515625, -0.02685546875, -0.048065185546875, 0.017547607421875, -0.0175018310546875, -0.0248870849609375, 0.052642822265625, -0.0253753662109375, -0.02215576171875, -0.01171112060546875, -0.0247955322265625, 0.018310546875, -0.005161285400390625, -0.010040283203125, 0.020294189453125, -0.004436492919921875, -0.03228759765625, -0.0626220703125, -0.0020732879638671875, 0.00920867919921875, -0.041015625, 0.00411224365234375, 

### QUERY ENGINE 

- Create your index 
- This just connects the llamaindex to the current Chroma vector store where our 108 nodes + embeddings are stored

In [27]:

index = VectorStoreIndex.from_vector_store(vector_store)



- checking how many gemini models my api is capable of 

In [ ]:


client = genai.Client()

for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-

- setting up the LLM we are using gemini

In [45]:

config = types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(thinking_budget=0),
    max_output_tokens=512,
    temperature=1,
)

llm = GoogleGenAI(model="gemini-3.1-flash-lite-preview",generation_config=config)


In [46]:
"""
    Your Question
"How many parameters LLaMA 2 model has?"
        │
        ▼
  OpenAI Embedding          ← embeds YOUR question into a vector
  [0.023, 0.041, ...]
        │
        ▼
  Chroma Vector Store       ← compares question vector vs 108 node vectors
        │
        ▼
  Top 5 similar nodes       ← similarity_top_k=5
  (chunks that talk about LLaMA 2)
        │
        ▼
  Gemini 2.5 Flash          ← receives question + 5 chunks as context
        │
        ▼
  res.response              ← "LLaMA 2 has 7B, 13B, and 70B parameter versions

  
"""

'\n    Your Question\n"How many parameters LLaMA 2 model has?"\n        │\n        ▼\n  OpenAI Embedding          ← embeds YOUR question into a vector\n  [0.023, 0.041, ...]\n        │\n        ▼\n  Chroma Vector Store       ← compares question vector vs 108 node vectors\n        │\n        ▼\n  Top 5 similar nodes       ← similarity_top_k=5\n  (chunks that talk about LLaMA 2)\n        │\n        ▼\n  Gemini 2.5 Flash          ← receives question + 5 chunks as context\n        │\n        ▼\n  res.response              ← "LLaMA 2 has 7B, 13B, and 70B parameter versions\n\n  \n'

In [47]:
query_engine = index.as_query_engine(llm=llm, similarity_top_k=5)

res = query_engine.query("How many parameters LLaMA 2 model has?")



In [48]:
print(res.response)

Llama 2 is available in four different model sizes: 7 billion, 13 billion, 34 billion, and 70 billion parameters.


In [49]:
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Title\t", src.metadata["title"])
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

Node ID	 9afbdeaea403deb0f61cdc3bca5b4a96afe98f4166b36b4f8606cc41a7c0a4c1
Title	 Fine-Tuning a Llama-2 7B Model for Python Code Generation
Text	 only fine-tuning a small number of additional parameters, with virtually all model parameters remaining frozen. PEFT has been found to produce good generalization with relatively low-volume datasets. Furthermore, it enhances the reusability and portability of the model, as the small checkpoints obtained can be easily added to the base model, and the base model can be easily fine-tuned and reused in multiple scenarios by adding the PEFT parameters. Finally, since the base model is not adjusted, all the knowledge acquired in the pre-training phase is preserved, thus avoiding catastrophic forgetting. Most widely used PEFT techniques aim to keep the pre-trained base model untouched and add new layers or parameters on top of it. These layers are called "Adapters" and the technique of their adjustment "adapter-tuning", we add these layers to the pre

## EVALS

- we will evaluate the retrieval process and quality of the anwers

We can evaluate our RAG system with a dataset of questions and associated chunks. Given a question, we can see if the RAG system retrieves the correct chunks of text that can answer the question.

You can generate a synthetic dataset with an LLM such as `gemini-3.1-flash-lite-preview` or create an authentic and manually curated dataset.

Note that a **well curated dataset will always be a better option**, especially for a specific domain or use case.

- In our example, we will generate a synthetic dataset using `gemini-3.1-flash-lite-preview` to make it simple.
- This is the default prompt that the `generate_question_context_pairs` function will uses:

In [56]:
# We can also load the dataset from a previously saved json file.

"""

from llama_index.core.evaluation import EmbeddingQAFinetuneDataset

rag_eval_dataset = EmbeddingQAFinetuneDataset.from_json("./rag_eval_dataset.json")

"""

'\n\nfrom llama_index.core.evaluation import EmbeddingQAFinetuneDataset\n\nrag_eval_dataset = EmbeddingQAFinetuneDataset.from_json("./rag_eval_dataset.json")\n\n'

In [50]:
DEFAULT_QA_GENERATE_PROMPT_TMPL = """\
Context information is below.

---------------------
{context_str}
---------------------

Given the context information and no prior knowledge,
generate only questions based on the below query.

You are a Teacher/Professor. Your task is to setup \
{num_questions_per_chunk} questions for an upcoming \
quiz/examination. The questions should be diverse in nature \
across the document. Restrict the questions to the \
context information provided."
"""

In [53]:
def generate_question_context_pairs(
    nodes: List[TextNode],
    llm: LLM,
    qa_generate_prompt_tmpl: str = DEFAULT_QA_GENERATE_PROMPT_TMPL,
    num_questions_per_chunk: int = 2,
    request_delay: float = 2.0
) -> EmbeddingQAFinetuneDataset:
    """Generate examples given a set of nodes with delays between requests."""
    node_dict = {
        node.node_id: node.get_content(metadata_mode=MetadataMode.NONE)
        for node in nodes
    }

    queries = {}
    relevant_docs = {}

    for node_id, text in tqdm(node_dict.items()):
        query = qa_generate_prompt_tmpl.format(
            context_str=text, num_questions_per_chunk=num_questions_per_chunk
        )
        response = llm.complete(query)

        result = str(response).strip().split("\n")
        questions = [
            re.sub(r"^\d+[\).\s]", "", question).strip() for question in result
        ]
        questions = [question for question in questions if len(question) > 0][
            :num_questions_per_chunk
        ]

        num_questions_generated = len(questions)
        if num_questions_generated < num_questions_per_chunk:
            warnings.warn(
                f"Fewer questions generated ({num_questions_generated}) "
                f"than requested ({num_questions_per_chunk})."
            )

        for question in questions:
            question_id = str(uuid.uuid4())
            queries[question_id] = question
            relevant_docs[question_id] = [node_id]

        time.sleep(request_delay)

    return EmbeddingQAFinetuneDataset(
        queries=queries, corpus=node_dict, relevant_docs=relevant_docs
    )

In [55]:
#from llama_index.core.evaluation import generate_question_context_pairs

# Define a query engine that is responsible for retrieving related pieces of text,
# and using a LLM to formulate the final answer.

from llama_index.llms.google_genai import GoogleGenAI

llm = GoogleGenAI(model="gemini-3.1-flash-lite-preview", temperature=0.3)

rag_eval_dataset = generate_question_context_pairs(
    nodes[:25],
    llm=llm,
    num_questions_per_chunk=1,
    request_delay=4
)

# Save the dataset as a json file for later use
rag_eval_dataset.save_json("/Users/r31881/Desktop/AI/helloai/data/rag_eval_dataset.json")

100%|██████████| 25/25 [01:57<00:00,  4.71s/it]


- second way of generating the questions bit pricy but fast 

In [ ]:
# #Paid-Gemini API Key

# from llama_index.core.evaluation import generate_question_context_pairs

# #Define a query engine that is responsible for retrieving related pieces of text,
# #and using a LLM to formulate the final answer.

# from llama_index.llms.google_genai import GoogleGenAI

# import google.genai.types as types

# config = types.GenerateContentConfig(
#     thinking_config=types.ThinkingConfig(thinking_budget=0),
#     max_output_tokens=512,
#     temperature=0.3,
# )

# llm = GoogleGenAI(model="gemini-2.5-flash", generation_config=config)

# rag_eval_dataset = generate_question_context_pairs(nodes, llm=llm, num_questions_per_chunk=1)

# # We can save the dataset as a json file for later use.
# rag_eval_dataset.save_json("./rag_eval_dataset.json")

### EVALUATION

### Evaluation for Hit Rate and Mean Reciprocal Rank (MRR)

We will make use of `RetrieverEvaluator` available in Llama-index. We will measure the Hit Rate and Mean Reciprocal Rank (MRR).

**Hit Rate:**

Think of the Hit Rate like playing a game of guessing. You're given a question and you need to guess the correct answer from a list of options. The Hit Rate measures how often you guess the correct answer by only looking at your top few guesses. If you often find the right answer in your first few guesses, you have a high Hit Rate. So, in the context of a retrieval system, it's about how frequently the system finds the correct document within its top 'k' picks (where 'k' is a number you decide, like top 5 or top 10).

**Mean Reciprocal Rank (MRR):**

MRR is a bit like measuring how quickly you can find a treasure in a list of boxes. Imagine you have a row of boxes and only one of them has a treasure. The MRR calculates how close to the start of the row the treasure box is, on average. If the treasure is always in the first box you open, you're doing great and have an MRR of 1. If it's in the second box, the score is 1/2, since you took two tries to find it. If it's in the third box, your score is 1/3, and so on. MRR averages these scores across all your searches. So, for a retrieval system, MRR looks at where the correct document ranks in the system's guesses. If it's usually near the top, the MRR will be high, indicating good performance.



- In summary, Hit Rate tells you how often the system gets it right in its top guesses, and MRR tells you how close to the top the right answer usually is. Both metrics are useful for evaluating the effectiveness of a retrieval system, like how well a search engine or a recommendation system works.

In [59]:
def display_results_retriever(name, eval_results):
    """Display results from evaluate."""

    metric_dicts = []
    for eval_result in eval_results:
        metric_dict = eval_result.metric_vals_dict
        metric_dicts.append(metric_dict)

    full_df = pd.DataFrame(metric_dicts)

    hit_rate = full_df["hit_rate"].mean()
    mrr = full_df["mrr"].mean()

    metric_df = pd.DataFrame(
        {"Retriever Name": [name], "Hit Rate": [hit_rate], "MRR": [mrr]}
    )

    return metric_df

In [61]:
# We can evaluate the retievers with different top_k values.
for i in [2, 4, 6, 8, 10]:
    retriever = index.as_retriever(similarity_top_k=i)
    retriever_evaluator = RetrieverEvaluator.from_metric_names(
        ["mrr", "hit_rate"], retriever=retriever
    )
    eval_results = await retriever_evaluator.aevaluate_dataset(
        rag_eval_dataset, workers=32
    )
    print(display_results_retriever(f"Retriever top_{i}", eval_results))

time.sleep(60)

    Retriever Name  Hit Rate   MRR
0  Retriever top_2      0.32  0.28
    Retriever Name  Hit Rate   MRR
0  Retriever top_4      0.36  0.29
    Retriever Name  Hit Rate    MRR
0  Retriever top_6       0.4  0.298
    Retriever Name  Hit Rate       MRR
0  Retriever top_8      0.48  0.308714
     Retriever Name  Hit Rate       MRR
0  Retriever top_10      0.52  0.312714


### Evaluation using Relevance and Faithfulness metrics.

Here, we evaluate the answer generated by the LLM. Is the answer using the correct context? Is the answer faithful to the context? Is the answer relevant to the question?

An LLM will answer these questions, more specifically `gpt-5`.

**`FaithfulnessEvaluator`**
Evaluates if the answer is faithful to the retrieved contexts (in other words, whether there's an hallucination).

**`RelevancyEvaluator`**
Evaluates whether the retrieved context and answer are relevant to the user question.

Now, let's see how the top_k value affects these two metrics.

In [63]:
# Define an LLM as a judge
llm_gpt5 = OpenAI(model="gpt-5", additional_kwargs={'reasoning_effort':'minimal'})
llm_gpt5_mini = OpenAI(model="gpt-5-mini", additional_kwargs={'reasoning_effort':'minimal'})

# Initiate the faithfulnes and relevancy evaluator objects
faithfulness_evaluator = FaithfulnessEvaluator(llm=llm_gpt5)
relevancy_evaluator = RelevancyEvaluator(llm=llm_gpt5)

# Extract the questions from the dataset
queries = list(rag_eval_dataset.queries.values())
# Limit to first 10 question to save time (!!remove this line in production!!)
batch_eval_queries = queries[:20]

# The batch evaluator runs the evaluation in batches
runner = BatchEvalRunner(
    {"faithfulness": faithfulness_evaluator, "relevancy": relevancy_evaluator},
    workers=32,
)


# Define a for-loop to try different `similarity_top_k` values
for i in [2, 4, 6, 8, 10]:
    # Set query engine with different number of returned chunks
    query_engine = index.as_query_engine(similarity_top_k=i, llm = llm_gpt5_mini)

    # Run the evaluation
    eval_results = await runner.aevaluate_queries(query_engine, queries=batch_eval_queries)

    # Printing the results
    faithfulness_score = sum(
        result.passing for result in eval_results["faithfulness"]
    ) / len(eval_results["faithfulness"])
    print(f"top_{i} faithfulness_score: {faithfulness_score}")

    relevancy_score = sum(result.passing for result in eval_results["relevancy"]) / len(
        eval_results["relevancy"]
    )
    print(f"top_{i} relevancy_score: {relevancy_score}")
    print("="*15)

top_2 faithfulness_score: 0.5
top_2 relevancy_score: 0.75
top_4 faithfulness_score: 0.65
top_4 relevancy_score: 1.0
top_6 faithfulness_score: 0.65
top_6 relevancy_score: 0.95
top_8 faithfulness_score: 0.5
top_8 relevancy_score: 1.0
top_10 faithfulness_score: 0.55
top_10 relevancy_score: 1.0


- CORRECTNESS METRICS

In [66]:
query = (
    "Can you explain the theory of relativity proposed by Albert Einstein in" " detail?"
)

reference = """
Certainly! Albert Einstein's theory of relativity consists of two main components: special relativity and general relativity. Special relativity, published in 1905, introduced the concept that the laws of physics are the same for all non-accelerating observers and that the speed of light in a vacuum is a constant, regardless of the motion of the source or observer. It also gave rise to the famous equation E=mc², which relates energy (E) and mass (m).

General relativity, published in 1915, extended these ideas to include the effects of gravity. According to general relativity, gravity is not a force between masses, as described by Newton's theory of gravity, but rather the result of the warping of space and time by mass and energy. Massive objects, such as planets and stars, cause a curvature in spacetime, and smaller objects follow curved paths in response to this curvature. This concept is often illustrated using the analogy of a heavy ball placed on a rubber sheet, causing it to create a depression that other objects (representing smaller masses) naturally move towards.

In essence, general relativity provided a new understanding of gravity, explaining phenomena like the bending of light by gravity (gravitational lensing) and the precession of the orbit of Mercury. It has been confirmed through numerous experiments and observations and has become a fundamental theory in modern physics.
"""

response = """
Certainly! Albert Einstein's theory of relativity consists of two main components: special relativity and general relativity. Special relativity, published in 1905, introduced the concept that the laws of physics are the same for all non-accelerating observers and that the speed of light in a vacuum is a constant, regardless of the motion of the source or observer. It also gave rise to the famous equation E=mc², which relates energy (E) and mass (m).

However, general relativity, published in 1915, extended these ideas to include the effects of magnetism. According to general relativity, gravity is not a force between masses but rather the result of the warping of space and time by magnetic fields generated by massive objects. Massive objects, such as planets and stars, create magnetic fields that cause a curvature in spacetime, and smaller objects follow curved paths in response to this magnetic curvature. This concept is often illustrated using the analogy of a heavy ball placed on a rubber sheet with magnets underneath, causing it to create a depression that other objects (representing smaller masses) naturally move towards due to magnetic attraction.
"""

In [67]:
evaluator = CorrectnessEvaluator(llm=llm_gpt5)

result = evaluator.evaluate(query=query,response=response,reference=reference,)

In [68]:
result.score

1.5

In [69]:
result.feedback

'The answer starts correctly about special relativity but then makes a major error: it claims general relativity involves effects of magnetism and that magnetic fields from massive objects warp spacetime. In reality, GR describes gravity as spacetime curvature due to mass-energy (stress-energy tensor), not magnetism. The rubber sheet analogy with magnets is incorrect. Therefore, while partially relevant, it is largely incorrect.'